
# Workflow Practice

In this notebook, you’ll practice connecting to a SQLite database, creating tables from CSV files using Pandas, and writing SQL queries to explore the data.

The dataset comes from the [Bike Store Sample Database](https://www.kaggle.com/datasets/dillonmyrick/bike-store-sample-database) by Dillon Myrick. It models a fictional bike retailer with multiple stores, products, customers, and staff. Each table connects to others using foreign keys such as `customer_id`, `store_id`, and `product_id`.

You’ll:
- Connect to a local SQLite database
- Create tables using `pandas.to_sql()`
- Write and test SQL queries using `pd.read_sql()`

All of your work will take place directly in this notebook. Each question prompt is written below as a Markdown cell, followed by an empty code cell for you to write your query.



## Step 1: Connect to the Database

Run the following cell to connect to (or create) a SQLite database called `bike_store.db`.  
If the file doesn’t exist yet, SQLite will automatically create it.


In [1]:
import sqlite3
import pandas as pd
import glob

In [2]:
connection = sqlite3.connect("bike_store.db")
connection


## Step 2: Create Tables from CSV Files

The `data/` folder contains one CSV file per table.  
Use `pandas.read_csv()` and `DataFrame.to_sql()` to load each file into your database.

You only need to do this once.  
After that, you’ll be able to run queries against your newly created tables.


In [3]:
# # Example for one file
# customers = pd.read_csv("data/customers.csv")
# customers.to_sql("customers", connection, if_exists="replace", index=False)

In [4]:
# Repeat for all other files in the data folder, or use a loop.
csv_list = glob.glob('./data/*.csv')        # create list of data files to convert
for f in csv_list:                          # for each filename...
    db_name = f.replace('./data/','')       # ...remove the leading './data/' and...
    db_name = db_name.replace('.csv','')    # ...remove the trailing '.csv' for filename sans path or extension.
    working_file = pd.read_csv(f)           # working through the file list, choose file.
    working_file.to_sql(db_name, connection, if_exists='replace', index=False)  # commit each csv file to its own table in bike_store.db

### Verify Your Tables

Run a query to make sure your tables were created successfully.

In [5]:

pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", connection)


,name
0,customers
1,products
2,categories
3,stores
4,orders
5,staffs
6,stocks
7,order_items
8,brands


## Step 3: Test a Simple Query

Before starting the exercises, confirm your connection and tables are working by previewing the first few rows of the `customers` table.

In [6]:

pd.read_sql("SELECT * FROM customers LIMIT 10", connection)


,customer_id,first_name,last_name,phone,email,street,city,state,zip_code
0,1,Debra,Burks,NaN,debra.burks@yahoo.com,9273 Thorne Ave.,Orchard Park,NY,14127
1,2,Kasha,Todd,NaN,kasha.todd@yahoo.com,910 Vine Street,Campbell,CA,95008
2,3,Tameka,Fisher,NaN,tameka.fisher@aol.com,769C Honey Creek St.,Redondo Beach,CA,90278
3,4,Daryl,Spence,NaN,daryl.spence@aol.com,988 Pearl Lane,Uniondale,NY,11553
4,5,Charolette,Rice,(916) 381-6003,charolette.rice@msn.com,107 River Dr.,Sacramento,CA,95820
5,6,Lyndsey,Bean,NaN,lyndsey.bean@hotmail.com,769 West Road,Fairport,NY,14450
6,7,Latasha,Hays,(716) 986-3359,latasha.hays@hotmail.com,7014 Manor Station Rd.,Buffalo,NY,14215
7,8,Jacquline,Duncan,NaN,jacquline.duncan@yahoo.com,15 Brown St.,Jackson Heights,NY,11372
8,9,Genoveva,Baldwin,NaN,genoveva.baldwin@msn.com,8550 Spruce Drive,Port Washington,NY,11050
9,10,Pamelia,Newman,NaN,pamelia.newman@gmail.com,476 Chestnut Ave.,Monroe,NY,10950


### Q1. List all customers and their cities.

Return the first name, last name, and city of each customer. Sort alphabetically by last name and then by first name.

In [7]:
# Your query here
query1 = "SELECT first_name, last_name, city FROM customers ORDER BY last_name, first_name" 
pd.read_sql(query1, connection)

,first_name,last_name,city
0,Ester,Acevedo,San Lorenzo
1,Jamika,Acevedo,Ozone Park
2,Penny,Acevedo,Ballston Spa
3,Bettyann,Acosta,Lancaster
4,Shery,Acosta,Saratoga Springs
...,...,...,...
1440,Edda,Young,North Tonawanda
1441,Jasmin,Young,Helotes
1442,Alexandria,Zamora,Schenectady
1443,Jayme,Zamora,Springfield Gardens


### Q2. Show all products and their prices.

Display each product name along with its list price. Sort by price in descending order.

In [8]:
# Your query here
query2 = "SELECT product_name, list_price FROM products ORDER BY list_price DESC" 
pd.read_sql(query2, connection)

,product_name,list_price
0,Trek Domane SLR 9 Disc - 2018,11999.99
1,Trek Domane SLR 8 Disc - 2018,7499.99
2,Trek Silque SLR 8 Women's - 2017,6499.99
3,Trek Domane SL Frameset - 2018,6499.99
4,Trek Domane SL Frameset Women's - 2018,6499.99
...,...,...
316,Trek Kickster - 2018,159.99
317,Trek Boy's Kickster - 2015/2017,149.99
318,Trek Girl's Kickster - 2017,149.99
319,Sun Bicycles Lil Kitt'n - 2017,109.99


### Q3. Find all customers from California.

Return first name, last name, city, and state for all customers whose state is 'CA'. Sort alphabetically by last name.

In [9]:
# Your query here
query3 = "SELECT first_name, last_name, city, state FROM customers WHERE state=='CA' ORDER BY last_name ASC" 
pd.read_sql(query3, connection)

,first_name,last_name,city,state
0,Ester,Acevedo,San Lorenzo,CA
1,Jamaal,Albert,Torrance,CA
2,Sindy,Anderson,Pomona,CA
3,Twana,Arnold,Anaheim,CA
4,Selene,Austin,Duarte,CA
...,...,...,...,...
279,Darren,Witt,Coachella,CA
280,Lucy,Woods,Palos Verdes Peninsula,CA
281,Joel,Wynn,San Diego,CA
282,Yvone,Yates,San Pablo,CA


### Q4. Count how many products are in each category.

Return the category name and the number of products in that category. Sort from the highest count to the lowest.

In [10]:
# Your query here
query4 = "SELECT c.category_name, COUNT(p.category_id) AS category_count" \
"   FROM products p" \
"   JOIN categories c" \
"   ON p.category_id = c.category_id" \
"   GROUP BY p.category_id" \
"   ORDER BY category_count DESC"
pd.read_sql(query4, connection)

,category_name,category_count
0,Cruisers Bicycles,78
1,Road Bikes,60
2,Mountain Bikes,60
3,Children Bicycles,59
4,Comfort Bicycles,30
5,Electric Bikes,24
6,Cyclocross Bicycles,10


### Q5. Find all orders placed in 2018.

List the order ID, order date, and customer ID for orders made during the year 2018. Sort by order date.

In [32]:
# Your query here
query5 = "WITH order_year (order_date)" \
"   AS (" \
"       SELECT order_date " \
"       FROM orders" \
"       WHERE EXTRACT (YEAR FROM DATE(order_date))=2018" \
"   )" \
"" \
"SELECT DATE(order_date) AS order_year" \
"   FROM orders" 
pd.read_sql(query5, connection)

DatabaseError: Execution failed on sql 'WITH order_year (order_date)   AS (       SELECT order_date        FROM orders       WHERE EXTRACT (YEAR FROM DATE(order_date))=2018   )SELECT DATE(order_date) AS order_year   FROM orders': near "FROM": syntax error

### Q6. Show each order with its total number of items.

Join the `orders` and `order_items` tables. Group by order ID and return the number of items per order.

In [ ]:
# Your query here
query6 = """ 
"""
pd.read_sql(query6, connection)

### Q7. List total revenue per store.

Revenue = quantity * list_price * (1 - discount). Join `orders`, `order_items`, and `stores`, group by store name, and return total revenue.

In [ ]:
# Your query here
query7 = """ 
"""
pd.read_sql(query7, connection)

### Q8. Find the top 5 customers who spent the most overall.

Join `customers`, `orders`, and `order_items`. Sum the total spending per customer and return the top five spenders.

In [ ]:
# Your query here
query8 = """ 
"""
pd.read_sql(query8, connection)

### Q9. Show the best-selling product in each category.

Join `products`, `order_items`, and `categories`. For each category, identify the product with the highest total quantity sold.

In [ ]:
# Your query here
query9 = """ 
"""
pd.read_sql(query9, connection)

### Q10. Identify the employees (staff) who processed the most orders.

Join `staffs` and `orders`. Count the number of orders handled by each staff member and return the results sorted by highest total.

In [ ]:
# Your query here
query10 = """ 
"""
pd.read_sql(query10, connection)